In [1]:
import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# ============================================================
# FUNCTION 4 - WEEK 11 BAYESIAN OPTIMISATION
# Run from inside week11/
# ============================================================
#
# Strategy:
# - Refit ARD Matern GP including Week 10.
# - Check Week 10 realised calibration.
# - Centre local search on the ACTUAL best observation.
# - Include moderate wider/global candidates as diagnostics.
# - Compare posterior mean, EI and UCB.
#
# F4 has now produced two strong observations in the same
# local basin, so we do not need the defensive treatment used
# for Function 3.
# ============================================================


# ------------------------------------------------------------
# 1. Load cumulative Week 11 data
# ------------------------------------------------------------

X = np.load("function4/initial_inputs.npy")
Y = np.load("function4/initial_outputs.npy").reshape(-1)

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("================================")
print("DATA")
print("================================")

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(best_x, "->", best_y)

print("\nY range:")
print("min =", Y.min())
print("max =", Y.max())
print("std =", Y.std())


# ------------------------------------------------------------
# 2. Week 10 calibration check
# ------------------------------------------------------------
#
# Week 10 selected:
# [0.36684455, 0.40962296, 0.43083265, 0.41965779]
#
# Prior prediction:
# mean ≈ 0.404105
# std  ≈ 0.266878
#
# Actual:
# 0.6208031832357475
# ------------------------------------------------------------

week10_pred_mean = 0.404105
week10_pred_std = 0.266878
week10_actual = 0.6208031832357475

week10_error = (
    week10_actual
    - week10_pred_mean
)

week10_z_error = (
    week10_error
    / week10_pred_std
)

print("\n================================")
print("WEEK 10 CALIBRATION CHECK")
print("================================")

print("Predicted mean:", week10_pred_mean)
print("Predicted std :", week10_pred_std)
print("Actual        :", week10_actual)

print("\nPrediction error:")
print(week10_error)

print("\nError / predicted std:")
print(week10_z_error)


# ------------------------------------------------------------
# 3. Fit ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(1e-3, 1e3)
    )
    *
    Matern(
        length_scale=np.ones(4) * 0.2,
        length_scale_bounds=(0.01, 2.0),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=30,
    random_state=42
)

gp.fit(X, Y)

print("\n================================")
print("GP FIT")
print("================================")

print("\nFitted kernel:")
print(gp.kernel_)

lengthscales = (
    gp.kernel_.k1.k2.length_scale
)

inverse_ls = (
    1.0 / lengthscales
)

relative_sensitivity = (
    inverse_ls
    / inverse_ls.sum()
)

print("\nARD lengthscales:")
print(lengthscales)

print(
    "\nNormalised inverse-lengthscale sensitivity:"
)
print(relative_sensitivity)


# ------------------------------------------------------------
# 4. Expected Improvement
# ------------------------------------------------------------

def expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
):

    improvement = (
        mu - best_y - xi
    )

    valid = sigma > 1e-12

    Z = np.zeros_like(mu)

    Z[valid] = (
        improvement[valid]
        / sigma[valid]
    )

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid]
        * norm.cdf(Z[valid])
        +
        sigma[valid]
        * norm.pdf(Z[valid])
    )

    return EI


# ------------------------------------------------------------
# 5. Candidate generation
# ------------------------------------------------------------

rng = np.random.default_rng(42)

local_scale = np.clip(
    0.20 * lengthscales,
    0.015,
    0.08
)

wide_scale = np.clip(
    0.40 * lengthscales,
    0.04,
    0.15
)

print("\n================================")
print("CANDIDATE SCALES")
print("================================")

print("Local widths:")
print(local_scale)

print("\nWide widths:")
print(wide_scale)


# Dense local search around actual incumbent

local_candidates = (
    best_x
    + rng.normal(
        0,
        local_scale,
        size=(160000, 4)
    )
)

# Moderate wider exploration around same basin

wide_candidates = (
    best_x
    + rng.normal(
        0,
        wide_scale,
        size=(110000, 4)
    )
)

# Global diagnostic pool

global_candidates = rng.uniform(
    0,
    1,
    size=(90000, 4)
)

local_candidates = np.clip(
    local_candidates,
    0,
    1
)

wide_candidates = np.clip(
    wide_candidates,
    0,
    1
)

candidates = np.vstack([
    local_candidates,
    wide_candidates,
    global_candidates
])


# ------------------------------------------------------------
# 6. Remove near-duplicates
# ------------------------------------------------------------

tree = cKDTree(X)

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.008
]

print("\nCandidates after duplicate filtering:")
print(len(candidates))


# ------------------------------------------------------------
# 7. GP predictions
# ------------------------------------------------------------

mu, sigma = gp.predict(
    candidates,
    return_std=True
)


# ------------------------------------------------------------
# 8. Primary EI
# ------------------------------------------------------------

EI = expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
)

ei_idx = np.argmax(EI)

print("\n================================")
print("PRIMARY EI")
print("================================")

print("candidate =", candidates[ei_idx])
print("mean =", mu[ei_idx])
print("std =", sigma[ei_idx])
print("EI =", EI[ei_idx])


# ------------------------------------------------------------
# 9. EI sensitivity
# ------------------------------------------------------------

y_scale = np.std(Y)

xi_values = [
    0.0,
    0.01 * y_scale,
    0.05 * y_scale,
    0.10 * y_scale
]

print("\n================================")
print("EI SENSITIVITY")
print("================================\n")

for xi in xi_values:

    EI_test = expected_improvement(
        mu,
        sigma,
        best_y,
        xi=xi
    )

    idx = np.argmax(EI_test)

    print(
        "xi =", f"{xi:.6e}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n EI =", round(EI_test[idx], 8),
        "\n"
    )


# ------------------------------------------------------------
# 10. Highest predicted mean
# ------------------------------------------------------------

mean_idx = np.argmax(mu)

print("\n================================")
print("HIGHEST PREDICTED MEAN")
print("================================")

print("candidate =", candidates[mean_idx])
print("mean =", mu[mean_idx])
print("std =", sigma[mean_idx])


# ------------------------------------------------------------
# 11. UCB diagnostics
# ------------------------------------------------------------

print("\n================================")
print("UCB DIAGNOSTICS")
print("================================\n")

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n UCB =", round(UCB[idx], 6),
        "\n"
    )


# ------------------------------------------------------------
# 12. Distance from actual incumbent
# ------------------------------------------------------------

print("\n================================")
print("DISTANCE FROM CURRENT BEST")
print("================================")

print(
    "EI:",
    np.linalg.norm(
        candidates[ei_idx]
        - best_x
    )
)

print(
    "Highest mean:",
    np.linalg.norm(
        candidates[mean_idx]
        - best_x
    )
)

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(UCB)

    print(
        f"UCB beta={beta}:",
        np.linalg.norm(
            candidates[idx]
            - best_x
        )
    )


# ------------------------------------------------------------
# 13. Domain boundary check
# ------------------------------------------------------------

def domain_boundary_status(
    x,
    tol=0.01
):

    status = []

    for j in range(len(x)):

        if x[j] <= tol:
            status.append(
                f"x{j+1}~0"
            )

        elif x[j] >= 1.0 - tol:
            status.append(
                f"x{j+1}~1"
            )

    if not status:
        return "interior"

    return ", ".join(status)


print("\n================================")
print("DOMAIN BOUNDARY CHECK")
print("================================")

print(
    "EI:",
    domain_boundary_status(
        candidates[ei_idx]
    )
)

print(
    "Highest mean:",
    domain_boundary_status(
        candidates[mean_idx]
    )
)

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(UCB)

    print(
        f"UCB beta={beta}:",
        domain_boundary_status(
            candidates[idx]
        )
    )

DATA
X shape: (40, 4)
Y shape: (40,)

Current best:
[0.361985 0.41201  0.421437 0.427725] -> 0.6704983808455585

Y range:
min = -32.625660215962455
max = 0.6704983808455585
std = 9.68579329286326

WEEK 10 CALIBRATION CHECK
Predicted mean: 0.404105
Predicted std : 0.266878
Actual        : 0.6208031832357475

Prediction error:
0.21669818323574747

Error / predicted std:
0.8119746971865327


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 32 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)



GP FIT

Fitted kernel:
2.61**2 * Matern(length_scale=[1.66, 1.47, 1.38, 1.48], nu=2.5) + WhiteKernel(noise_level=0.000572)

ARD lengthscales:
[1.65871282 1.46965028 1.38120922 1.47583648]

Normalised inverse-lengthscale sensitivity:
[0.22454392 0.25343028 0.26965783 0.25236798]

CANDIDATE SCALES
Local widths:
[0.08 0.08 0.08 0.08]

Wide widths:
[0.15 0.15 0.15 0.15]

Candidates after duplicate filtering:
359988

PRIMARY EI
candidate = [0.36975291 0.38882167 0.42649841 0.41455296]
mean = 0.42497543044604313
std = 0.2810022820453932
EI = 0.029605560525272828

EI SENSITIVITY

xi = 0.000000e+00 
 candidate = [0.36975291 0.38882167 0.42649841 0.41455296] 
 mean = 0.424975 
 std = 0.281002 
 EI = 0.02960556 

xi = 9.685793e-02 
 candidate = [0.36975291 0.38882167 0.42649841 0.41455296] 
 mean = 0.424975 
 std = 0.281002 
 EI = 0.01517827 

xi = 4.842897e-01 
 candidate = [0.38667706 0.35930058 0.41765629 0.41116358] 
 mean = 0.246482 
 std = 0.356912 
 EI = 0.00062186 

xi = 9.685793e-01 
 

In [2]:
# ============================================================
# FINAL FUNCTION 4 - WEEK 11 SELECTION
# ============================================================
#
# Week 10 calibration was reasonable (~ +0.81 sigma).
#
# Highest posterior mean and ALL tested UCB beta values
# select exactly the same candidate.
#
# EI moves farther from the incumbent and trades away
# predicted mean for extra uncertainty.
#
# Therefore select the stable local consensus point.

final_idx = np.argmax(mu)

week11_candidate = candidates[final_idx]

print("Week 11 Function 4 candidate:")
print(week11_candidate)

print("\nPredicted mean:")
print(mu[final_idx])

print("\nPredicted std:")
print(sigma[final_idx])

print("\nDistance from current best:")
print(
    np.linalg.norm(
        week11_candidate - best_x
    )
)

portal = "-".join(
    f"{x:.6f}"
    for x in week11_candidate
)

print("\nPortal format:")
print(portal)


Week 11 Function 4 candidate:
[0.37167674 0.4037974  0.42658414 0.41934237]

Predicted mean:
0.4535070043533622

Predicted std:
0.25854242580580267

Distance from current best:
0.016066679526986375

Portal format:
0.371677-0.403797-0.426584-0.419342
